# Zomato Restaurant Data Analysis

**Team:** Harsh Mehlana & Fenil Parmar  
**Course:** CSE, Semester 5 — Indus University

## Objective
Analyze restaurant distribution, cuisines, ratings, pricing, online delivery, table booking, and customer voting patterns using Pandas, NumPy, Matplotlib, and Seaborn.

## 1. Load libraries and dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

local_path = '../data/zomato.csv'
if os.path.exists(local_path):
    data_path = local_path
else:
    import kagglehub
    data_dir = kagglehub.dataset_download('shrutimehta/zomato-restaurants-data')
    data_path = os.path.join(data_dir, 'zomato.csv')

df = pd.read_csv(data_path, encoding='latin-1')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. Data inspection and cleaning

Only 9 cuisine values are missing. Rating `0` is treated as **not rated**, not as a zero-star rating. Rating-based analysis therefore uses `rated_df`.

In [ ]:
print('Missing values:')
print(df.isna().sum().sort_values(ascending=False).head())
print('Duplicate rows:', df.duplicated().sum())

df['Cuisines'] = df['Cuisines'].fillna('Unknown')
rated_df = df[df['Aggregate rating'] > 0].copy()

print('Restaurants:', len(df))
print('Rated restaurants:', len(rated_df))
print('Not rated:', (df['Aggregate rating'] == 0).sum())

## 3. Restaurants by city

In [ ]:
city_counts = df['City'].value_counts().head(10)
plt.figure(figsize=(10,6))
plt.barh(city_counts.index[::-1], city_counts.values[::-1])
plt.title('Top 10 Cities by Number of Restaurants')
plt.xlabel('Number of Restaurants')
plt.tight_layout()
plt.show()
display(city_counts.to_frame('restaurant_count'))

## 4. Most common cuisines

In [ ]:
cuisine_counts = df['Cuisines'].str.split(', ').explode().value_counts().head(15)
plt.figure(figsize=(10,7))
plt.barh(cuisine_counts.index[::-1], cuisine_counts.values[::-1])
plt.title('Top 15 Most Common Cuisines')
plt.xlabel('Restaurant Mentions')
plt.tight_layout()
plt.show()
display(cuisine_counts.to_frame('restaurant_mentions'))

## 5. Rating distribution

In [ ]:
status = pd.Series({'Rated': (df['Aggregate rating'] > 0).sum(), 'Not rated': (df['Aggregate rating'] == 0).sum()})
plt.figure(figsize=(7,5))
plt.bar(status.index, status.values)
plt.title('Rated vs Not-Rated Restaurants')
plt.ylabel('Number of Restaurants')
plt.show()

plt.figure(figsize=(9,5))
sns.histplot(rated_df['Aggregate rating'], bins=20, kde=True)
plt.title('Distribution of Ratings (Rated Restaurants)')
plt.xlabel('Aggregate Rating')
plt.show()
print('Mean rated score:', round(rated_df['Aggregate rating'].mean(),2))
print('Median rated score:', round(rated_df['Aggregate rating'].median(),2))

## 6. Price range

In [ ]:
price_counts = df['Price range'].value_counts().sort_index()
plt.figure(figsize=(8,5))
plt.bar(price_counts.index.astype(str), price_counts.values)
plt.title('Restaurant Distribution by Price Range')
plt.xlabel('Price Range')
plt.ylabel('Number of Restaurants')
plt.show()
display(price_counts.to_frame('restaurant_count'))

## 7. Cost vs rating

In [ ]:
sample = rated_df.sample(min(3000, len(rated_df)), random_state=42)
plt.figure(figsize=(10,6))
sns.scatterplot(data=sample, x='Average Cost for two', y='Aggregate rating', alpha=0.45)
plt.title('Average Cost for Two vs Restaurant Rating')
plt.show()
print('Correlation:', round(rated_df['Average Cost for two'].corr(rated_df['Aggregate rating']),3))

## 8. Online delivery vs rating

In [ ]:
delivery = rated_df.groupby('Has Online delivery')['Aggregate rating'].agg(['count','mean','median'])
display(delivery)
plt.figure(figsize=(8,5))
sns.barplot(x=delivery.index, y=delivery['mean'])
plt.title('Average Rating by Online Delivery Availability')
plt.xlabel('Has Online Delivery')
plt.ylabel('Average Rating')
plt.ylim(0,5)
plt.show()

## 9. Table booking vs rating

In [ ]:
booking = rated_df.groupby('Has Table booking')['Aggregate rating'].agg(['count','mean','median'])
display(booking)
plt.figure(figsize=(8,5))
sns.barplot(x=booking.index, y=booking['mean'])
plt.title('Average Rating by Table Booking Availability')
plt.xlabel('Has Table Booking')
plt.ylabel('Average Rating')
plt.ylim(0,5)
plt.show()

## 10. Votes vs rating

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=sample, x='Votes', y='Aggregate rating', alpha=0.45)
plt.xscale('log')
plt.title('Customer Votes vs Restaurant Rating')
plt.xlabel('Votes (log scale)')
plt.show()
print('Correlation:', round(rated_df['Votes'].corr(rated_df['Aggregate rating']),3))

## 11. Rating by price range and correlation heatmap

In [ ]:
price_rating = rated_df.groupby('Price range')['Aggregate rating'].mean()
display(price_rating.to_frame('average_rating'))
plt.figure(figsize=(8,5))
sns.barplot(x=price_rating.index, y=price_rating.values)
plt.title('Average Rating by Price Range')
plt.ylim(0,5)
plt.show()

corr = rated_df[['Average Cost for two','Price range','Aggregate rating','Votes']].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', square=True)
plt.title('Correlation Heatmap')
plt.show()

## 12. Key findings and conclusion

- New Delhi has the largest restaurant representation, followed by Gurgaon and Noida.
- North Indian is the most frequently mentioned cuisine, followed by Chinese and Fast Food.
- 2,148 records have rating 0 and are treated as not rated.
- Price range 1 is the most common category.
- Online-delivery restaurants have a higher average rating in this dataset than restaurants without online delivery.
- Table-booking restaurants have a higher average rating than restaurants without table booking.
- Votes have a moderate positive association with rating; average cost has a very weak linear association with rating.
- These are dataset associations, not proof of causation.

**Conclusion:** The project demonstrates a complete exploratory data-analysis workflow: data loading, inspection, cleaning, visualization, interpretation, and conclusion.